In [ ]:
import sys
import pandas as pd

print("Python path:", sys.executable)
print("Pandas version:", pd.__version__)


In [ ]:
patients_df = pd.read_csv(
    "/mnt/c/Users/vetts/Downloads/MimicIII/"
    "mimic-iii-clinical-database-1.4/PATIENTS.csv/PATIENTS.csv"
)

patients_df.head()

In [ ]:
print("Rows and columns:", patients_df.shape)

patients_df.info()

In [ ]:
admissions_df = pd.read_csv(
    "/mnt/c/Users/vetts/Downloads/MimicIII/"
    "mimic-iii-clinical-database-1.4/ADMISSIONS.csv/ADMISSIONS.csv"
)

print("Rows and columns:", admissions_df.shape)
admissions_df.info()

In [ ]:
admissions_df.head()

In [ ]:
admissions_df.head().T

In [ ]:
icustays_df = pd.read_csv(
    "/mnt/c/Users/vetts/Downloads/MimicIII/"
    "mimic-iii-clinical-database-1.4/ICUSTAYS.csv/ICUSTAYS.csv"
)

print("Rows and columns:", icustays_df.shape)
icustays_df.info()

In [ ]:
icustays_df.head().T

# 01 — Data Overview

## Objective
Explore the core tables for an ICU risk prediction project.

## Environment
Created a project-specific Python environment and verified pandas.

## Tables Reviewed
- PATIENTS: 46,520 rows and 8 columns.
- ADMISSIONS: 58,976 rows and 19 columns.
- ICUSTAYS: 61,532 rows and 12 columns.

## Initial Findings
- SUBJECT_ID identifies a patient.
- HADM_ID identifies a hospital admission.
- ICUSTAY_ID identifies an ICU stay.
- A patient can have multiple admissions.
- Date columns were loaded as strings.
- OUTTIME and LOS each have 10 missing values in ICUSTAYS.
- LOS represents ICU length of stay in days.

## Core Table Merge
Created icu_base_df with one row per ICU stay.

- Checked the primary identifiers for missing values and duplicates.
- Combined selected columns using validated left joins.
- Preserved all 61,532 ICU stays.
- The merged table contains 13 columns.
- Every ICU stay matched a patient and hospital admission.
- The original DataFrames were kept unchanged.

In [ ]:
for name, df, key in [
    ("PATIENTS", patients_df, "SUBJECT_ID"),
    ("ADMISSIONS", admissions_df, "HADM_ID"),
    ("ICUSTAYS", icustays_df, "ICUSTAY_ID"),
]:
    print(
        name,
        "| Missing IDs:", df[key].isna().sum(),
        "| Duplicate IDs:", df[key].duplicated().sum()
    )

In [ ]:
print(
    "ICU rows without a patient match:",
    (~icustays_df["SUBJECT_ID"].isin(patients_df["SUBJECT_ID"])).sum()
)

print(
    "ICU rows without an admission match:",
    (~icustays_df["HADM_ID"].isin(admissions_df["HADM_ID"])).sum()
)


In [ ]:
# 1. Select the required ICU stay columns
icu_base_df = icustays_df[
    [
        "SUBJECT_ID", "HADM_ID", "ICUSTAY_ID",
        "INTIME", "OUTTIME", "FIRST_CAREUNIT", "LOS"
    ]
].copy()

# 2. Add patient information
icu_base_df = icu_base_df.merge(
    patients_df[["SUBJECT_ID", "GENDER", "DOB"]],
    on="SUBJECT_ID",
    how="left",
    validate="many_to_one"
)

# 3. Add hospital admission information
icu_base_df = icu_base_df.merge(
    admissions_df[
        [
            "SUBJECT_ID", "HADM_ID",
            "ADMITTIME", "DISCHTIME",
            "ADMISSION_TYPE", "HOSPITAL_EXPIRE_FLAG"
        ]
    ],
    on=["SUBJECT_ID", "HADM_ID"],
    how="left",
    validate="many_to_one"
)

print("Merged table shape:", icu_base_df.shape)

In [ ]:
icu_base_df.head().T

In [ ]:
icu_base_df.head()

In [ ]:
print("Missing patient matches:", icu_base_df["DOB"].isna().sum())
print("Missing admission matches:", icu_base_df["ADMITTIME"].isna().sum())

## Core Table Merge

Selected columns from PATIENTS, ADMISSIONS, and ICUSTAYS were
combined into `icu_base_df`, with one row per ICU stay.

- Checked identifiers for missing values and duplicates.
- Used left joins with many-to-one validation.
- Preserved all 61,532 ICU stays, producing 13 columns.
- Confirmed that every ICU stay matched a patient and hospital admission.
- Kept the original DataFrames unchanged.

In [ ]:
date_columns = [
    "DOB", "ADMITTIME", "DISCHTIME", "INTIME", "OUTTIME"
]

for column in date_columns:
    icu_base_df[column] = pd.to_datetime(
        icu_base_df[column],
        errors="raise"
    )

icu_base_df[date_columns].dtypes

In [ ]:
icu_base_df["AGE_RAW"] = (
    (icu_base_df["INTIME"] - icu_base_df["DOB"])
    .dt.total_seconds()
    / (365.25 * 24 * 60 * 60)
)

icu_base_df["AGE_RAW"].describe()

In [ ]:
icu_base_df["AGE_RAW"].describe()

In [ ]:
icu_base_df["AGE_90_PLUS"] = (
    icu_base_df["AGE_RAW"] >= 90
).astype(int)

icu_base_df["AGE"] = icu_base_df["AGE_RAW"].clip(upper=90)

print("ICU stays in the 90+ group:",
      icu_base_df["AGE_90_PLUS"].sum())

icu_base_df["AGE"].describe()

In [ ]:
icu_base_df[["AGE_RAW", "AGE", "AGE_90_PLUS"]].head(10)

In [ ]:
icu_base_df.loc[
    icu_base_df["AGE_90_PLUS"] == 1,
    ["AGE_RAW", "AGE", "AGE_90_PLUS"]
].head()

## Age Preparation

- Converted birth, hospital admission/discharge, and ICU entry/exit
  columns to datetime.
- Calculated approximate age at ICU entry using INTIME (ICU entry)
  and DOB (date of birth).
- Preserved the original calculation in AGE_RAW.
- MIMIC-III masks ages over 89 by shifting birth dates, which can
  produce calculated ages above 300.
- Created AGE by capping calculated ages at 90.
  The value 90 represents the 90+ group, not an exact age.
- Created AGE_90_PLUS: 1 for calculated ages of 90 or above,
  and 0 otherwise.
- Identified 2,721 ICU stays in the 90+ group.
- No records were removed; all 61,532 ICU stays were retained.
- AGE_RAW will not be used as a model input.

In [ ]:
under_18_patient_count=icu_base_df.loc[icu_base_df["AGE"]<18,"SUBJECT_ID"].nunique()

print("Unique patients under 18:", under_18_patient_count)

In [ ]:
pediatric_icu_df=icu_base_df.loc[icu_base_df["AGE"]<18].copy()

print("Pediatric Icu Stays:",len(pediatric_icu_df))
print("Unique Pediatric Patients:",pediatric_icu_df["SUBJECT_ID"].nunique())

In [ ]:
adult_icu_df=icu_base_df.loc[icu_base_df["AGE"]>=18].copy()

In [ ]:
adult_icu_df.head()

In [ ]:
print (len(adult_icu_df))
print (adult_icu_df["SUBJECT_ID"].nunique())

In [ ]:
print (adult_icu_df.isna().sum())

In [ ]:
print (pediatric_icu_df.isna().sum())

In [ ]:
pediatric_icu_df=pediatric_icu_df.dropna(subset=["OUTTIME","LOS"])
adult_icu_df=adult_icu_df.dropna(subset=["OUTTIME","LOS"])

### Age Preparation and Cohort Separation

- Calculated age at ICU admission using the date of birth and ICU admission time.
- Preserved the calculated age in `AGE_RAW`, capped `AGE` at 90, and added `AGE_90_PLUS`. A value of 90 represents the 90+ age group, not an exact age.
- Created separate pediatric (`AGE < 18`) and adult (`AGE >= 18`) DataFrames.
- Before removing missing values:
  - **Pediatric cohort:** 8,200 ICU stays from 7,967 unique patients.
  - **Adult cohort:** 53,332 ICU stays from 38,512 unique patients.
- Removed rows with missing ICU discharge time (`OUTTIME`) or length of stay (`LOS`) from both cohort DataFrames.
- Kept the original merged table, `icu_base_df`, unchanged.